# 1. Импорт и инициализация библиотек

Загрузка всех необходимых библиотек для работы с данными, машинным обучением и визуализацией

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymorphy3
import re
import warnings
import nltk
import xgboost as xgb
import time

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

morph = pymorphy3.MorphAnalyzer(lang='ru')

# 2. Загрузка и подготовка данных

Чтение данных из CSV файла и первичный осмотр структуры

In [ ]:
df = pd.read_csv("./movies_final_cleaned.csv", index_col=0)
print(f"Данные загружены: {df.shape[0]} фильмов, {df.shape[1]} колонок")
print(f"Память: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 2.1 Базовый анализ данных

In [ ]:
print("Первые 5 строк:")
df.head()

In [ ]:
print("\nИнформация о типах данных:")
df.info()

In [ ]:
print("\nСтатистика числовых признаков:")
df.describe()

In [ ]:
print("\nСтатистика текстовых признаков:")
df.describe(include='O')

In [ ]:
print("\nДистрибуция числовых признаков:")
df.hist(figsize=(20, 10), bins=30)
plt.tight_layout()
plt.show()

## 2.2 Подготовка стоп-слов и инструментов обработки текста

In [ ]:
try:
    stopwords.words('russian')
    print("Стоп-слова уже загружены")
except LookupError:
    print("Загружаю стоп-слова...")
    nltk.download('stopwords')

russian_stopwords = set(stopwords.words('russian'))
print(f"Загружено {len(russian_stopwords)} русских стоп-слов")

# 3. Стандартизация и нормализация данных

Приведение данных к единому формату и подготовка к обучению моделей

## 3.1 Унификация названий стран

In [ ]:
country_map = {
    'United States': 'США',
    'USA': 'США',
    'United Kingdom': 'Великобритания',
    'Japan': 'Япония',
    'Italy': 'Италия',
    'New Zealand': 'Новая Зеландия',
    'France': 'Франция',
    'South Korea': 'Южная Корея',
    'India': 'Индия',
    'Brazil': 'Бразилия'
}

df['country'] = df['country'].replace(country_map)
print(f"Уникальные страны после унификации: {df['country'].nunique()}")
print(df['country'].value_counts())

## 3.2 Обработка нулевых рейтингов

In [ ]:
null_rating_count = (df['rating_kp'] == 0).sum()
print(f"Найдено {null_rating_count} нулевых рейтингов Кинопоиска")

df.loc[df['rating_kp'] == 0, 'rating_kp'] = df.loc[df['rating_kp'] == 0, 'rating_imdb']

print(f"После обработки нулевых рейтингов осталось: {(df['rating_kp'] == 0).sum()}")

## 3.3 Преобразование жанров в бинарные признаки

In [ ]:
df['genres_list'] = df['genres'].apply(
    lambda x: [g.strip().lower() for g in x.split(',')]
)

print(f"Пример списка жанров:")
print(df['genres_list'].iloc[0])

In [ ]:
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(df['genres_list'])
genres_df = pd.DataFrame(
    genres_encoded, 
    columns=mlb.classes_, 
    index=df.index
)

print(f"Уникальные жанры: {len(mlb.classes_)}")
print(f"Жанры: {list(mlb.classes_)[:10]}...")

In [ ]:
df_final = pd.concat([df, genres_df], axis=1)
print(f"Объединённый датафрейм: {df_final.shape}")

## 3.4 Подготовка числовых признаков

In [ ]:
features_to_scale = ['year', 'rating_kp', 'rating_imdb', 'votes_imdb']
all_features = features_to_scale + list(mlb.classes_)

print(f"Всего числовых признаков: {len(all_features)}")
print(f"Основные признаки: {features_to_scale}")
print(f"Жанровые признаки: {len(mlb.classes_)}")

In [ ]:
X = df_final[all_features]
print(f"Матрица признаков: {X.shape}")

In [ ]:
scaler_initial = MinMaxScaler()
X_scaled = scaler_initial.fit_transform(X)
X_scaled_df = pd.DataFrame(
    X_scaled, 
    columns=all_features, 
    index=df.index
)

print("Масштабирование завершено")
print(f"Признаков для обучения: {X_scaled_df.shape[1]}")
print(f"Диапазон значений: [{X_scaled_df.min().min():.4f}, {X_scaled_df.max().max():.4f}]")

## 3.5 Анализ корреляций основных метрик

In [ ]:
correlation_data = df[['year', 'rating_kp', 'rating_imdb', 'votes_imdb']]
corr_matrix = correlation_data.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, square=True)
plt.title('Корреляция основных метрик фильмов')
plt.tight_layout()
plt.show()

print("Матрица корреляций:")
print(corr_matrix)

## 3.6 Создание целевой переменной и подготовка данных для обучения

In [ ]:
df_final['is_hit'] = (df_final['rating_imdb'] >= 8.0).astype(int)

hit_count = (df_final['is_hit'] == 1).sum()
not_hit_count = (df_final['is_hit'] == 0).sum()

print(f"Распределение классов:")
print(f"  Хиты (рейтинг >= 8.0): {hit_count} фильмов ({hit_count/len(df_final)*100:.1f}%)")
print(f"  Обычные (рейтинг < 8.0): {not_hit_count} фильмов ({not_hit_count/len(df_final)*100:.1f}%)")

plt.figure(figsize=(8, 6))
df_final['is_hit'].value_counts().plot(kind='bar', color=['#FF6B6B', '#4ECDC4'])
plt.title('Распределение целевого класса')
plt.xlabel('Класс (0 - обычный, 1 - хит)')
plt.ylabel('Количество')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
drop_cols = [
    'title', 'country', 'genres', 'description', 
    'genres_list', 'rating_kp', 'rating_imdb', 'is_hit'
]

X = df_final.drop(columns=drop_cols)
y = df_final['is_hit']

print(f"Целевые данные подготовлены")
print(f"X: {X.shape}")
print(f"y: {y.shape}")

In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print(f"Признаки масштабированы")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Разделение на train/test завершено")
print(f"Обучающая выборка: {X_train.shape}")
print(f"Тестовая выборка: {X_test.shape}")
print(f"\nРаспределение классов в train: {dict(pd.Series(y_train).value_counts())}")
print(f"Распределение классов в test: {dict(pd.Series(y_test).value_counts())}")

# 4. Обработка текстовых данных

Полная пpipeline обработки описаний фильмов: очистка, лемматизация, TF-IDF векторизация

## 4.1 Функции для обработки текста

In [ ]:
def clean_text(text):
    """
    Очистка текста от спецсимволов и приведение к нижнему регистру
    
    Parameters:
    -----------
    text : str
        Исходный текст
        
    Returns:
    --------
    str
        Очищенный текст
    """
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r'[^а-яёa-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


def lemmatize_text(text):
    """
    Лемматизация текста и удаление стоп-слов
    
    Parameters:
    -----------
    text : str
        Очищенный текст
        
    Returns:
    --------
    str
        Лемматизированный текст
    """
    if not text:
        return ""
    
    words = text.split()
    lemmas = []
    
    for word in words:
        lemma = morph.parse(word)[0].normal_form
        if lemma not in russian_stopwords and len(lemma) > 2:
            lemmas.append(lemma)
    
    return ' '.join(lemmas)


def process_descriptions(data, verbose=True):
    """
    Полная обработка описаний фильмов
    
    Parameters:
    -----------
    data : pd.DataFrame
        Датафрейм с колонкой 'description'
    verbose : bool
        Выводить ли прогресс обработки
        
    Returns:
    --------
    pd.DataFrame
        Датафрейм с добавленными колонками 'description_clean' и 'description_lemma'
    """
    data_copy = data.copy()
    
    if verbose:
        print("Очистка текстов...")
    data_copy['description_clean'] = data_copy['description'].apply(clean_text)
    
    if verbose:
        print("Лемматизация...")
    data_copy['description_lemma'] = data_copy['description_clean'].apply(lemmatize_text)
    
    return data_copy


print("Функции обработки текста определены")

## 4.2 Применение функций обработки

In [ ]:
df = process_descriptions(df, verbose=True)

print(f"\nОбработка завершена")
print(f"Пример обработки:")

for i in range(min(2, len(df))):
    print(f"\nФильм {i+1}: {df['title'].iloc[i]}")
    print(f"  Оригинал: {df['description'].iloc[i][:100]}...")
    print(f"  Очищен: {df['description_clean'].iloc[i][:100]}...")
    print(f"  Лемма: {df['description_lemma'].iloc[i][:100]}...")

In [ ]:
df_final['description_lemma'] = df['description_lemma'].copy()

print("Лемматизированный текст добавлен в df_final")

## 4.3 TF-IDF векторизация

In [ ]:
print("Инициализация TF-IDF векторизатора...")

tfidf = TfidfVectorizer(
    max_features=100,
    min_df=3,
    max_df=0.85,
    ngram_range=(1, 2),
    strip_accents='unicode'
)

print("Применение TF-IDF...")
tfidf_matrix = tfidf.fit_transform(df['description_lemma'])

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f'tfidf_{name}' for name in tfidf.get_feature_names_out()],
    index=df.index
)

print(f"\nTF-IDF матрица создана: {tfidf_df.shape}")

In [ ]:
feature_freq = tfidf_df.sum(axis=0).sort_values(ascending=False)

print("Топ-20 наиболее важных слов:")
for i, (feature, freq) in enumerate(feature_freq.head(20).items(), 1):
    clean_name = feature.replace('tfidf_', '')
    print(f"{i:2d}. {clean_name:20s}: {freq:.4f}")

plt.figure(figsize=(12, 6))
feature_freq.head(15).plot(kind='barh', color='steelblue')
plt.title('Топ-15 слов по частоте встречаемости (TF-IDF)')
plt.xlabel('Сумма TF-IDF значений')
plt.tight_layout()
plt.show()

## 4.4 Визуализация облаков слов

In [ ]:
high_rated_text = ' '.join(
    df_final[df_final['is_hit'] == 1]['description_lemma'].dropna()
)
low_rated_text = ' '.join(
    df_final[df_final['is_hit'] == 0]['description_lemma'].dropna()
)

print(f"Статистика текстовых данных:")
print(f"  Хиты: {(df_final['is_hit'] == 1).sum()} фильмов")
print(f"  Обычные: {(df_final['is_hit'] == 0).sum()} фильмов")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

if high_rated_text.strip():
    wc_high = WordCloud(
        width=800, 
        height=400, 
        background_color='white',
        colormap='Greens', 
        max_words=80
    ).generate(high_rated_text)
    axes[0].imshow(wc_high, interpolation='bilinear')
else:
    axes[0].text(0.5, 0.5, 'Нет текста', ha='center', va='center')

axes[0].set_title('Облако слов: Хиты (рейтинг >= 8.0)', fontsize=14, fontweight='bold')
axes[0].axis('off')

if low_rated_text.strip():
    wc_low = WordCloud(
        width=800, 
        height=400, 
        background_color='white',
        colormap='Reds', 
        max_words=80
    ).generate(low_rated_text)
    axes[1].imshow(wc_low, interpolation='bilinear')
else:
    axes[1].text(0.5, 0.5, 'Нет текста', ha='center', va='center')

axes[1].set_title('Облако слов: Обычные (рейтинг < 8.0)', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("Облака слов созданы")

## 4.5 Объединение всех признаков

In [ ]:
print("Объединение признаков...\n")

X_combined = pd.concat([X, tfidf_df], axis=1)

print(f"Статистика объединения:")
print(f"  Численные признаки: {X.shape[1]}")
print(f"  TF-IDF признаки: {tfidf_df.shape[1]}")
print(f"  Всего признаков: {X_combined.shape[1]}")
print(f"  Образцы: {X_combined.shape[0]}")

In [ ]:
print("Масштабирование объединённых признаков...\n")

scaler_combined = MinMaxScaler()
X_scaled_combined = scaler_combined.fit_transform(X_combined)

print(f"Масштабирование завершено")
print(f"  Min значение: {X_scaled_combined.min():.4f}")
print(f"  Max значение: {X_scaled_combined.max():.4f}")
print(f"  Mean значение: {X_scaled_combined.mean():.4f}")

In [ ]:
print("Разделение данных на обучающую и тестовую выборки...\n")

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_scaled_combined, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Разделение завершено")
print(f"\nОбучающая выборка:")
print(f"  Размер: {X_train_full.shape}")
print(f"  Класс 0: {(y_train_full == 0).sum()}")
print(f"  Класс 1: {(y_train_full == 1).sum()}")

print(f"\nТестовая выборка:")
print(f"  Размер: {X_test_full.shape}")
print(f"  Класс 0: {(y_test_full == 0).sum()}")
print(f"  Класс 1: {(y_test_full == 1).sum()}")

# 5. Обучение моделей машинного обучения

Подготовка и обучение 4 различных моделей классификации

## 5.1 Вспомогательные функции для обучения

In [ ]:
def evaluate_model(y_true, y_pred, y_pred_proba, model_name="Model"):
    """
    Вычисление метрик качества модели
    """
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_pred_proba)
    }
    
    return metrics


def print_model_results(metrics, train_time):
    """
    Красивый вывод результатов модели
    """
    print(f"\nРезультаты модели:")
    print(f"  Accuracy:        {metrics['accuracy']:.4f}")
    print(f"  Precision:       {metrics['precision']:.4f}")
    print(f"  Recall:          {metrics['recall']:.4f}")
    print(f"  F1-Score:        {metrics['f1']:.4f}")
    print(f"  ROC-AUC:         {metrics['auc']:.4f}")
    print(f"  Время обучения:  {train_time:.2f} сек")


models_results = {}

print("Вспомогательные функции определены")

## 5.2 Модель 1: Логистическая регрессия

In [ ]:
print("\n" + "="*70)
print("Обучение модели 1: Логистическая регрессия")
print("="*70)

start_time = time.time()

lr_model = LogisticRegression(
    max_iter=1000, 
    random_state=42, 
    n_jobs=-1, 
    verbose=0
)
lr_model.fit(X_train_full, y_train_full)

train_time_lr = time.time() - start_time

In [ ]:
y_pred_lr = lr_model.predict(X_test_full)
y_pred_proba_lr = lr_model.predict_proba(X_test_full)[:, 1]

metrics_lr = evaluate_model(y_test_full, y_pred_lr, y_pred_proba_lr)
print_model_results(metrics_lr, train_time_lr)

print(f"\n" + "-"*70)
print(f"Classification Report:")
print("-"*70)
print(classification_report(y_test_full, y_pred_lr))

models_results['Логистическая регрессия'] = {
    'model': lr_model,
    'metrics': metrics_lr,
    'time': train_time_lr,
    'y_pred': y_pred_lr,
    'y_proba': y_pred_proba_lr
}

## 5.3 Модель 2: XGBoost (Gradient Boosting)

In [ ]:
print("\n" + "="*70)
print("Обучение модели 2: XGBoost")
print("="*70)

start_time = time.time()

xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    verbosity=0
)
xgb_model.fit(X_train_full, y_train_full)

train_time_xgb = time.time() - start_time

In [ ]:
y_pred_xgb = xgb_model.predict(X_test_full)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_full)[:, 1]

metrics_xgb = evaluate_model(y_test_full, y_pred_xgb, y_pred_proba_xgb)
print_model_results(metrics_xgb, train_time_xgb)

print(f"\n" + "-"*70)
print(f"Classification Report:")
print("-"*70)
print(classification_report(y_test_full, y_pred_xgb))

models_results['XGBoost'] = {
    'model': xgb_model,
    'metrics': metrics_xgb,
    'time': train_time_xgb,
    'y_pred': y_pred_xgb,
    'y_proba': y_pred_proba_xgb
}

In [ ]:
feature_importance_xgb = pd.Series(
    xgb_model.feature_importances_,
    index=X_combined.columns
).nlargest(15)

plt.figure(figsize=(10, 6))
feature_importance_xgb.plot(kind='barh', color='steelblue')
plt.title('XGBoost: Топ-15 важных признаков', fontsize=12, fontweight='bold')
plt.xlabel('Важность (Gain)')
plt.tight_layout()
plt.show()

## 5.4 Модель 3: Random Forest

In [ ]:
print("\n" + "="*70)
print("Обучение модели 3: Random Forest")
print("="*70)

start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rf_model.fit(X_train_full, y_train_full)

train_time_rf = time.time() - start_time

In [ ]:
y_pred_rf = rf_model.predict(X_test_full)
y_pred_proba_rf = rf_model.predict_proba(X_test_full)[:, 1]

metrics_rf = evaluate_model(y_test_full, y_pred_rf, y_pred_proba_rf)
print_model_results(metrics_rf, train_time_rf)

print(f"\n" + "-"*70)
print(f"Classification Report:")
print("-"*70)
print(classification_report(y_test_full, y_pred_rf))

models_results['Random Forest'] = {
    'model': rf_model,
    'metrics': metrics_rf,
    'time': train_time_rf,
    'y_pred': y_pred_rf,
    'y_proba': y_pred_proba_rf
}

In [ ]:
feature_importance_rf = pd.Series(
    rf_model.feature_importances_,
    index=X_combined.columns
).nlargest(15)

plt.figure(figsize=(10, 6))
feature_importance_rf.plot(kind='barh', color='forestgreen')
plt.title('Random Forest: Топ-15 важных признаков', fontsize=12, fontweight='bold')
plt.xlabel('Важность (MDI)')
plt.tight_layout()
plt.show()

## 5.5 Модель 4: Нейросеть (PyTorch)

In [ ]:
class MovieClassifier(nn.Module):
    """
    Архитектура глубокой нейросети для классификации фильмов
    """
    def __init__(self, input_dim):
        super(MovieClassifier, self).__init__()
        
        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(0.3)
        
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(0.3)
        
        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.dropout3 = nn.Dropout(0.2)
        
        self.fc4 = nn.Linear(64, 32)
        self.dropout4 = nn.Dropout(0.2)
        
        self.fc5 = nn.Linear(32, 16)
        self.fc6 = nn.Linear(16, 1)
        
    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)
        
        x = torch.relu(self.fc4(x))
        x = self.dropout4(x)
        
        x = torch.relu(self.fc5(x))
        x = torch.sigmoid(self.fc6(x))
        
        return x


print("Архитектура нейросети определена")

In [ ]:
print("\n" + "="*70)
print("Обучение модели 4: Нейросеть (PyTorch)")
print("="*70)

X_train_tensor = torch.FloatTensor(X_train_full).to(device)
y_train_tensor = torch.FloatTensor(y_train_full.values).reshape(-1, 1).to(device)
X_test_tensor = torch.FloatTensor(X_test_full).to(device)
y_test_tensor = torch.FloatTensor(y_test_full.values).reshape(-1, 1).to(device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"\nДанные подготовлены")
print(f"  Устройство: {device}")
print(f"  Размер батча: 32")

In [ ]:
print("\nКреирование нейросети...")

nn_model = MovieClassifier(X_train_full.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(
    nn_model.parameters(), 
    lr=0.001, 
    weight_decay=1e-5
)

print(f"Нейросеть готова к обучению")

In [ ]:
print("\nОбучение нейросети (50 эпох)...\n")

start_time = time.time()

epochs = 50
losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(epochs):
    nn_model.train()
    train_loss = 0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = nn_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(nn_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    nn_model.eval()
    with torch.no_grad():
        val_outputs = nn_model(X_train_tensor)
        val_loss = criterion(val_outputs, y_train_tensor)
    
    avg_train_loss = train_loss / len(train_loader)
    losses.append(avg_train_loss)
    val_losses.append(val_loss.item())
    
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= 5:
        print(f"Ранняя остановка на эпохе {epoch+1}")
        break
    
    if (epoch + 1) % 10 == 0:
        print(f"Эпоха {epoch+1:2d}/{epochs} | Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f}")

train_time_nn = time.time() - start_time

In [ ]:
nn_model.eval()
with torch.no_grad():
    y_pred_proba_nn = nn_model(X_test_tensor).cpu().numpy()
    y_pred_nn = (y_pred_proba_nn > 0.5).astype(int).flatten()

metrics_nn = evaluate_model(y_test_full, y_pred_nn, y_pred_proba_nn)
print_model_results(metrics_nn, train_time_nn)

print(f"\n" + "-"*70)
print(f"Classification Report:")
print("-"*70)
print(classification_report(y_test_full, y_pred_nn))

models_results['Нейросеть'] = {
    'model': nn_model,
    'metrics': metrics_nn,
    'time': train_time_nn,
    'y_pred': y_pred_nn,
    'y_proba': y_pred_proba_nn.flatten()
}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses, label='Обучение', linewidth=2)
axes[0].plot(val_losses, label='Валидация', linewidth=2)
axes[0].set_title('Динамика Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].fill_between(range(len(losses)), losses, alpha=0.3)
axes[1].plot(range(len(losses)), losses, linewidth=2)
axes[1].set_title('Кривая обучения', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 6. Сравнение и анализ моделей

Комплексное сравнение всех обученных моделей и их производительности

## 6.1 Таблица результатов

In [ ]:
print("\n" + "="*70)
print("Сравнение всех 4 моделей")
print("="*70)

results_df = pd.DataFrame({
    'Модель': list(models_results.keys()),
    'Accuracy': [models_results[m]['metrics']['accuracy'] for m in models_results],
    'Precision': [models_results[m]['metrics']['precision'] for m in models_results],
    'Recall': [models_results[m]['metrics']['recall'] for m in models_results],
    'F1-Score': [models_results[m]['metrics']['f1'] for m in models_results],
    'ROC-AUC': [models_results[m]['metrics']['auc'] for m in models_results],
    'Время (сек)': [models_results[m]['time'] for m in models_results]
})

print("\nРезультаты моделей:")
print(results_df.to_string(index=False))

In [ ]:
print("\nЛучшие модели по метрикам:")
print(f"\n  Accuracy:  {results_df.loc[results_df['Accuracy'].idxmax(), 'Модель']}")
print(f"             {results_df['Accuracy'].max():.4f}")

print(f"\n  Precision: {results_df.loc[results_df['Precision'].idxmax(), 'Модель']}")
print(f"             {results_df['Precision'].max():.4f}")

print(f"\n  Recall:    {results_df.loc[results_df['Recall'].idxmax(), 'Модель']}")
print(f"             {results_df['Recall'].max():.4f}")

print(f"\n  F1-Score:  {results_df.loc[results_df['F1-Score'].idxmax(), 'Модель']}")
print(f"             {results_df['F1-Score'].max():.4f}")

print(f"\n  ROC-AUC:   {results_df.loc[results_df['ROC-AUC'].idxmax(), 'Модель']}")
print(f"             {results_df['ROC-AUC'].max():.4f}")

print(f"\n  Скорость:  {results_df.loc[results_df['Время (сек)'].idxmin(), 'Модель']}")
print(f"             {results_df['Время (сек)'].min():.2f} сек")

## 6.2 Визуализация метрик качества

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Сравнение всех 4 моделей по метрикам', fontsize=16, fontweight='bold')

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for idx, metric in enumerate(metrics_list):
    ax = axes[idx // 3, idx % 3]
    bars = ax.bar(
        results_df['Модель'], 
        results_df[metric], 
        color=colors, 
        alpha=0.8, 
        edgecolor='black', 
        linewidth=1.5
    )
    ax.set_title(f'{metric}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Score')
    ax.set_ylim([0, 1])
    ax.axhline(
        y=results_df[metric].mean(), 
        color='red', 
        linestyle='--', 
        linewidth=2, 
        label='Mean'
    )
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=15)
    
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2., 
            height,
            f'{height:.3f}', 
            ha='center', 
            va='bottom', 
            fontweight='bold', 
            fontsize=9
        )

ax = axes[1, 2]
bars = ax.bar(
    results_df['Модель'], 
    results_df['Время (сек)'], 
    color=colors, 
    alpha=0.8, 
    edgecolor='black', 
    linewidth=1.5
)
ax.set_title('Время обучения', fontweight='bold', fontsize=12)
ax.set_ylabel('Время (секунды)')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=15)

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2., 
        height,
        f'{height:.2f}s', 
        ha='center', 
        va='bottom', 
        fontweight='bold', 
        fontsize=9
    )

plt.tight_layout()
plt.show()

## 6.3 ROC-AUC Кривые

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

colors_roc = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for idx, (model_name, model_data) in enumerate(models_results.items()):
    fpr, tpr, _ = roc_curve(y_test_full, model_data['y_proba'])
    auc_score = model_data['metrics']['auc']
    ax.plot(
        fpr, 
        tpr, 
        label=f'{model_name} (AUC={auc_score:.4f})', 
        linewidth=2.5, 
        color=colors_roc[idx]
    )

ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Случайный классификатор')
ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC-AUC Кривые всех моделей', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

## 6.4 Матрицы ошибок

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Матрицы ошибок (Confusion Matrices)', fontsize=16, fontweight='bold')

for idx, (model_name, model_data) in enumerate(models_results.items()):
    ax = axes[idx // 2, idx % 2]
    cm = confusion_matrix(y_test_full, model_data['y_pred'])
    
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues', 
        ax=ax, 
        cbar=False,
        xticklabels=['Обычный', 'Хит'], 
        yticklabels=['Обычный', 'Хит']
    )
    ax.set_title(f'{model_name}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Истинный класс')
    ax.set_xlabel('Предсказанный класс')

plt.tight_layout()
plt.show()

# 7. Выводы и рекомендации

Заключительный анализ результатов и рекомендации по использованию моделей

In [ ]:
print("\n" + "="*70)
print("ИТОГОВЫЙ АНАЛИЗ И ВЫВОДЫ")
print("="*70)

print(f"\nОбщая информация о данных:")
print(f"  Всего фильмов: {len(df)}")
print(f"  Хитов (рейтинг >= 8.0): {(y == 1).sum()} ({(y == 1).sum()/len(y)*100:.1f}%)")
print(f"  Обычных фильмов: {(y == 0).sum()} ({(y == 0).sum()/len(y)*100:.1f}%)")

print(f"\nСтруктура данных для обучения:")
print(f"  Всего признаков: {X_combined.shape[1]}")
print(f"  Численные признаки: {X.shape[1]}")
print(f"  Текстовые признаки (TF-IDF): {tfidf_df.shape[1]}")
print(f"  Размер обучающей выборки: {X_train_full.shape[0]}")
print(f"  Размер тестовой выборки: {X_test_full.shape[0]}")

print(f"\nОбзор производительности моделей:")
best_accuracy_model = results_df.loc[results_df['Accuracy'].idxmax(), 'Модель']
best_accuracy = results_df['Accuracy'].max()
print(f"  Лучшая по Accuracy: {best_accuracy_model} ({best_accuracy:.4f})")

best_f1_model = results_df.loc[results_df['F1-Score'].idxmax(), 'Модель']
best_f1 = results_df['F1-Score'].max()
print(f"  Лучшая по F1-Score: {best_f1_model} ({best_f1:.4f})")

best_auc_model = results_df.loc[results_df['ROC-AUC'].idxmax(), 'Модель']
best_auc = results_df['ROC-AUC'].max()
print(f"  Лучшая по ROC-AUC: {best_auc_model} ({best_auc:.4f})")

fastest_model = results_df.loc[results_df['Время (сек)'].idxmin(), 'Модель']
fastest_time = results_df['Время (сек)'].min()
print(f"  Самая быстрая: {fastest_model} ({fastest_time:.2f} сек)")

print(f"\nВремя обучения моделей:")
for idx, row in results_df.iterrows():
    print(f"  {row['Модель']:25s}: {row['Время (сек)']:7.2f} сек")

print(f"\n" + "="*70)